In [0]:
from pyspark.sql import functions as F
 
VOLUME = "/Volumes/nyc_taxi/bronze/raw_files"
 
def ingest_month(year: int, month: int):
    """Read one month of raw trip data and append it to the Bronze table."""
    filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
    path = f"{VOLUME}/{filename}"
 
    df = spark.read.parquet(path)
 
    df = (df
        .withColumn("_source_file",  F.lit(filename))
        .withColumn("_ingested_at",  F.current_timestamp())
        .withColumn("_source_year",  F.lit(year))
        .withColumn("_source_month", F.lit(month))
    )
 
    (df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("nyc_taxi.bronze.yellow_tripdata"))
 
    print(f"{year}-{month:02d}: appended {df.count():,} rows")
 
 
ingest_month(2024, 1)


In [0]:
df_bronze = spark.table("nyc_taxi.bronze.yellow_tripdata")

original_count = df_bronze.count()
display(df_bronze.limit(20))

appended_rows = df_bronze.filter(
    (F.col("_source_year") == 2024) & (F.col("_source_month") == 1)
).count()
print(f"Original row count before append: {original_count - appended_rows}")
print(f"Rows appended for 2024-01: {appended_rows}")

In [0]:
for m in range(1, 13):
    try:
        ingest_month(2024, m)
    except Exception as e:
        print(f"2024-{m:02d} FAILED: {e}")

In [0]:
from pyspark.sql import functions as F
 
VOLUME = "/Volumes/nyc_taxi/bronze/raw_files"
 
def ingest_month(year: int, month: int):
    """Read one month of raw trip data and append it to the Bronze table."""
    filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
    path = f"{VOLUME}/{filename}"
 
    df = spark.read.parquet(path)
 
    df = (df
        .withColumn("_source_file",  F.lit(filename))
        .withColumn("_ingested_at",  F.current_timestamp())
        .withColumn("_source_year",  F.lit(year))
        .withColumn("_source_month", F.lit(month))
    )
 
    (df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("nyc_taxi.bronze.yellow_tripdata"))
 
    print(f"{year}-{month:02d}: appended {df.count():,} rows")

In [0]:
for m in range(2, 13):
    try:
        ingest_month(2024, m)
    except Exception as e:
        print(f"2024-{m:02d} FAILED: {e}")

In [0]:
%sql
SELECT _source_year, _source_month, COUNT(*) AS row_count
FROM nyc_taxi.bronze.yellow_tripdata
GROUP BY _source_year, _source_month
ORDER BY _source_month;

In [0]:
%sql
DROP TABLE IF EXISTS nyc_taxi.bronze.yellow_tripdata;

In [0]:
for m in range(1, 13):
    try:
        ingest_month(2024, m)
    except Exception as e:
        print(f"2024-{m:02d} FAILED: {e}")